In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
from datasets import load_dataset
from collections import Counter

# Load the customer support ticket dataset from Hugging Face
ds = load_dataset("Tobi-Bueck/customer-support-tickets",
                   data_files="dataset-tickets-multi-lang-4-20k.csv")
df = ds["train"].to_pandas()

# Keep only English-language tickets (dataset also contains German tickets)
df = df[df["language"] == "en"].copy()
df = df.dropna(subset=["body", "queue"])
texts = df["body"].tolist()
labels = df["queue"].tolist()

print(f"Loaded {len(texts)} documents.")
print("Label distribution:", Counter(labels))

c:\Users\sande\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 11922 documents.
Label distribution: Counter({'Technical Support': 3412, 'Product Support': 2231, 'Customer Service': 1859, 'IT Support': 1391, 'Billing and Payments': 1302, 'Returns and Exchanges': 582, 'Service Outages and Maintenance': 442, 'Sales and Pre-Sales': 330, 'Human Resources': 205, 'General Inquiry': 168})


In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

8345 1788 1789


In [4]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

preds = clf.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"Macro F1: {f1_score(y_test, preds, average='macro'):.3f}")

Accuracy: 0.426
Macro F1: 0.289
